<a href="https://colab.research.google.com/github/HongDang99-Hust/Project_Embedded/blob/main/Yolov11test_hon_hop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Cài đặt thư viện Ultralytics
!pip install ultralytics

# Nhập thư viện và kiểm tra GPU
from ultralytics import YOLO
import ultralytics
import torch

ultralytics.checks()
print("GPU đang dùng:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Chưa có GPU! Hãy vào Runtime -> Change runtime type -> Chọn T4 GPU")

Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 43.1/112.6 GB disk)
GPU đang dùng: Tesla T4


In [ ]:
# 1. Kiểm tra GPU
!nvidia-smi

# 2. Cài đặt YOLOv12 và các thư viện hỗ trợ (Flash Attention giúp chạy nhanh hơn trên GPU)
!pip install -q git+https://github.com/sunsmarterjie/yolov12.git roboflow supervision flash-attn

Sat Apr 11 08:21:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             15W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
from google.colab import drive
drive.mount('/content/drive')

yaml_content = """
path: /content/dataset/dataset1_green
train: train/images
val: val/images
test: test/images

nc: 3
names: ['san_pham_du', 'thieu_dai_oc', 'thieu_long_den']
"""

import os
os.makedirs('/content/dataset/dataset', exist_ok=True)
with open('/content/dataset/dataset/dataset.yaml', 'w', encoding='utf-8') as f:
    f.write(yaml_content)
print("Đã thiết lập xong môi trường chuẩn!")

Mounted at /content/drive
Đã thiết lập xong môi trường chuẩn!


In [10]:
!mkdir -p /content/dataset
!unzip -q /content/drive/MyDrive/Do_An/dataset1_green.zip -d /content/dataset

In [ ]:
import ultralytics
ultralytics.checks()

Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 43.2/112.6 GB disk)


In [ ]:
# Cài đặt bản ổn định nhất của ultralytics (đã bao gồm YOLOv12)
!pip install -q ultralytics

In [ ]:
# Cài đặt YOLOv12 mà không cài flash-attn để tránh lỗi build
!pip install -q git+https://github.com/sunsmarterjie/yolov12.git roboflow supervision

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [12]:

!yolo task=segment mode=train model=yolo11s-seg.pt data="/content/dataset/dataset/dataset.yaml" epochs=100 imgsz=960 batch=16

Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/dataset/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, 

In [13]:
!yolo task=segment mode=predict model=/content/drive/MyDrive/Do_An/best2.pt source=/content/drive/MyDrive/Do_An/test_thu_3.jpg save=True

Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s-seg summary (fused): 114 layers, 10,067,977 parameters, 0 gradients, 32.8 GFLOPs

image 1/1 /content/drive/MyDrive/Do_An/test_thu_3.jpg: 960x960 (no detections), 34.4ms
Speed: 8.4ms preprocess, 34.4ms inference, 0.6ms postprocess per image at shape (1, 3, 960, 960)
Results saved to /content/runs/segment/predict5
💡 Learn more at https://docs.ultralytics.com/modes/predict


In [ ]:
# Thêm các tham số để ẩn khung chữ nhật và nhãn chữ

!yolo task=segment mode=predict model="/content/runs/segment/train/weights/best.pt" source="/content/runs/segment/predict/img_du6_rot_fliph.jpg" conf=0.5 save=True \
      show_boxes=False show_labels=False line_width=0

Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s-seg summary (fused): 114 layers, 10,067,977 parameters, 0 gradients, 32.8 GFLOPs

image 1/1 /content/runs/segment/predict/img_du6_rot_fliph.jpg: 640x640 2 san_pham_dus, 20.2ms
Speed: 2.2ms preprocess, 20.2ms inference, 20.6ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/segment/predict2
💡 Learn more at https://docs.ultralytics.com/modes/predict


In [ ]:
!cp /content/runs/segment/train/weights/best.pt /content/drive/MyDrive/Do_An/
print("Đã cất giữ bộ não thành công vào Drive!")

Đã cất giữ bộ não thành công vào Drive!


In [ ]:
from ultralytics import YOLO

# Tải model (yolov12s là bản nhẹ nhất)
model = YOLO('yolo12s.pt')

print("Chúc mừng! YOLOv12 đã cài đặt thành công.")

Chúc mừng! YOLOv12 đã cài đặt thành công.
